# Wikipedia AWQ calibration data: save one JSONL per language

每种语言保存为独立的 `<lang>.jsonl`，并在每条记录中保留 `language` 字段。这样 AWQ 的 multilingual、English-only 和 Arabic-only 实验不再依赖合并文件的隐含行顺序。

In [1]:
from pathlib import Path
import json
import pandas as pd
from tqdm.notebook import tqdm

LANGUAGES = ['ca', 'en', 'es', 'fr', 'hu', 'nl', 'tr', 'ar', 'el', 'he', 'ja', 'ko', 'zh']
SOURCE_ROOT = Path('/root/autodl-tmp/data/quantization/wikimedia')
OUTPUT_DIR = Path('/root/autodl-tmp/data/quantization/calibration_wikipedia_by_language')
SAMPLES_PER_LANGUAGE = 1000
SEED = 42

# 与原 notebook 一致，默认只从每种语言排序后的第一个 parquet 抽样。
# 如果确认内存充足，可改为 None 以读取该语言目录内的全部 parquet。
MAX_PARQUET_FILES = 1
WRITE_COMBINED_COPY = True
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR

PosixPath('/root/autodl-tmp/data/quantization/calibration_wikipedia_by_language')

In [2]:
def write_jsonl(records, output_path):
    tmp_path = output_path.with_suffix(output_path.suffix + '.tmp')
    with tmp_path.open('w', encoding='utf-8') as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + '\n')
    tmp_path.replace(output_path)

manifest = {
    'format_version': 1,
    'languages': LANGUAGES,
    'samples_per_language_requested': SAMPLES_PER_LANGUAGE,
    'seed': SEED,
    'source_root': str(SOURCE_ROOT),
    'files': {},
}

for lang in tqdm(LANGUAGES):
    parquet_files = sorted((SOURCE_ROOT / f'20231101.{lang}').glob('*.parquet'))
    if not parquet_files:
        raise FileNotFoundError(f'No parquet files found for {lang}')
    selected_files = parquet_files if MAX_PARQUET_FILES is None else parquet_files[:MAX_PARQUET_FILES]
    frames = [pd.read_parquet(path) for path in selected_files]
    df = pd.concat(frames, ignore_index=True)
    if 'text' not in df.columns:
        raise KeyError(f'Missing text column for {lang}')

    df = df.copy()
    df['text'] = df['text'].fillna('').astype(str).str.strip()
    df = df[df['text'].str.len() > 0]
    if len(df) < SAMPLES_PER_LANGUAGE:
        raise ValueError(f'{lang}: only {len(df)} non-empty records; need {SAMPLES_PER_LANGUAGE}')
    sampled = df.sample(n=SAMPLES_PER_LANGUAGE, replace=False, random_state=SEED)

    records = []
    for source_idx, row in sampled.iterrows():
        records.append({
            'sample_id': f'{lang}_{source_idx}',
            'language': lang,
            'source': 'wikimedia_20231101',
            'title': '' if pd.isna(row.get('title', '')) else str(row.get('title', '')),
            'text': str(row['text']),
        })

    output_path = OUTPUT_DIR / f'{lang}.jsonl'
    write_jsonl(records, output_path)
    manifest['files'][lang] = {
        'path': output_path.name,
        'records': len(records),
        'source_parquet_files': [str(path) for path in selected_files],
    }

manifest_tmp = OUTPUT_DIR / 'manifest.json.tmp'
manifest_tmp.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
manifest_tmp.replace(OUTPUT_DIR / 'manifest.json')
print(f'Saved per-language calibration files to {OUTPUT_DIR}')

  0%|          | 0/13 [00:00<?, ?it/s]

Saved per-language calibration files to /root/autodl-tmp/data/quantization/calibration_wikipedia_by_language


In [3]:
# 严格验证：文件名、language字段、数量和sample_id必须一致且无重复。
all_sample_ids = set()
validation = []
for lang in LANGUAGES:
    path = OUTPUT_DIR / f'{lang}.jsonl'
    rows = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
    wrong_language = [row['sample_id'] for row in rows if row.get('language') != lang]
    duplicate_ids = [row['sample_id'] for row in rows if row['sample_id'] in all_sample_ids]
    if wrong_language:
        raise ValueError(f'{lang}: language mismatch: {wrong_language[:5]}')
    if duplicate_ids:
        raise ValueError(f'{lang}: duplicate sample_id: {duplicate_ids[:5]}')
    if len(rows) != SAMPLES_PER_LANGUAGE:
        raise ValueError(f'{lang}: expected {SAMPLES_PER_LANGUAGE}, found {len(rows)}')
    if any(not str(row.get('text', '')).strip() for row in rows):
        raise ValueError(f'{lang}: empty text found')
    all_sample_ids.update(row['sample_id'] for row in rows)
    validation.append({'language': lang, 'records': len(rows), 'file': path.name})

pd.DataFrame(validation)

,language,records,file
0,ca,1000,ca.jsonl
1,en,1000,en.jsonl
2,es,1000,es.jsonl
3,fr,1000,fr.jsonl
4,hu,1000,hu.jsonl
5,nl,1000,nl.jsonl
6,tr,1000,tr.jsonl
7,ar,1000,ar.jsonl
8,el,1000,el.jsonl
9,he,1000,he.jsonl


In [4]:
# 可选：从已经验证过的语言文件流式生成一份兼容旧代码的合并副本。
if WRITE_COMBINED_COPY:
    combined_path = OUTPUT_DIR / 'calibration_wikipedia_multilingual.jsonl'
    combined_tmp = combined_path.with_suffix('.jsonl.tmp')
    with combined_tmp.open('w', encoding='utf-8') as out:
        for lang in LANGUAGES:
            with (OUTPUT_DIR / f'{lang}.jsonl').open('r', encoding='utf-8') as src:
                for line in src:
                    out.write(line)
    combined_tmp.replace(combined_path)
    print(f'Combined compatibility copy: {combined_path}')

print(f'Total validated samples: {len(all_sample_ids)}')

Combined compatibility copy: /root/autodl-tmp/data/quantization/calibration_wikipedia_by_language/calibration_wikipedia_multilingual.jsonl
Total validated samples: 13000
